<a href="https://colab.research.google.com/github/ximelovely/RecomendadorPeliculas/blob/main/Primer%20prueba%20de%20entrenamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets -q

In [2]:
from google.colab import files
uploaded = files.upload()

Saving prompts.jsonl to prompts.jsonl


In [3]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="prompts.jsonl", split="train")
print(dataset)  # Verifica que cargó bien

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 192
})


In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM

modelo = "datificate/gpt2-small-spanish"  # GPT2 entrenado en español
tokenizer = AutoTokenizer.from_pretrained(modelo)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(modelo)

config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/620 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/510M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: datificate/gpt2-small-spanish
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
def tokenizar(ejemplo):
    texto = ejemplo["prompt"] + " " + ejemplo["completion"]
    return tokenizer(texto, truncation=True, padding="max_length", max_length=256)

dataset_tokenizado = dataset.map(tokenizar)

Map:   0%|          | 0/192 [00:00<?, ? examples/s]

In [12]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir="./modelo_peliculas",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=100,
    logging_steps=10,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset_tokenizado,
    data_collator=data_collator,
)

trainer.train()

Step,Training Loss
10,4.605270
20,3.731069
30,3.523857
40,3.297495
50,3.150586
60,2.804956
70,2.692061
80,2.513248
90,2.581218
100,2.411013


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=144, training_loss=2.86690084139506, metrics={'train_runtime': 104.6263, 'train_samples_per_second': 5.505, 'train_steps_per_second': 1.376, 'total_flos': 109400593268736.0, 'train_loss': 2.86690084139506, 'epoch': 3.0})

In [14]:
from transformers import pipeline

generador = pipeline("text-generation", model=model, tokenizer=tokenizer)

resultado = generador(
    "Quiero algo romántico para ver con mi pareja",
    max_new_tokens=100,
    do_sample=True,        # Le da variedad a las respuestas
    temperature=0.7,       # Controla creatividad (0.1=robótico, 1.0=caótico)
    repetition_penalty=1.3, # Penaliza que repita frases
    top_p=0.9
)

print(resultado[0]["generated_text"])

Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Quiero algo romántico para ver con mi pareja. Te recomiendo El Tiempo de los Muertos (El Tiempo). Un chico descubre que su novia la ama, a pesar del amor y el desamor por el estilo visual principal. Posee un ritmo muy dinámico al interpretar algunos giros sobre las expectativas sociales en una película sin límites ni diálogos formales
Cuenta sus actuaciones musicales logrando una gran actuación cómica todo ello te haga llorar perfecto balanceando personajes inolvidables entre ellos mismo color intenso. Te recomenta El Tiempo de los Muertos: Una Película Interlutaantemente Sol
